<a href="https://colab.research.google.com/github/dave-heslop74/EMSC2010-W9-P1/blob/main/EMSC2010_W9_P1_NB2_uXXXXXXX_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EMSC2010-W9-P1-NB2-solution

### Soil temperature model with changing surface temperature

Simulate heat diffusing down through a column of soil with the surface temperature changing on a daily cycle with a mean ($T_{\mathrm{mean}}$) of 15°C and an amplitude ($A$) of 10°C.

I've put a copy of the code from ```EMSC2010-W9-P1-NB1.ipynb``` below. Modify the code to:
* Use soil 50 layers, each 5 cm thick.
* Set the initial surface and bottom temperatures to 15°C (the bottom temperature will not change during the model).
* Use a time step of 600 seconds (10 minutes).
* The period ($P$) of the surface temperature cycle is 86400 seconds (24 hours)
* Run the model for 20 days
* For each time step set the suface temperature according to the daily cycle:

$T_{\mathrm{surface}}(t) = T_{\mathrm{mean}} + A \sin (2 \pi t / P) $

To achieve this you will need to add this command into the code (you work out where) to allow the surface temperature to vary with time as a sine wave:

```T[0] = T_mean + A * np.sin(2 * np.pi * t / P)```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

We can define the size and properties of the soil column.

In [ ]:
num_layers = 50 # number of soil layers
dz = 0.05 # thickness of each layer (metres)
depth = num_layers * dz # total depth of column (metres)
kappa = 1e-6 # thermal diffusivity of the soil (m^2/s)

Next, we'll set up the initial temperture conditions and the boundary conditions.

In [ ]:
T_mean = 15.0 # mean temperature of daily cycle (°C)
A = 10.0 # amplitude of daily cycle (°C)
P = 60*60*24 #period of cycle (1 day in seconds)
T_initial = 15.0 # starting temperature throughout the soil (°C)
T_surface  = 15.0 # fixed surface temperature (°C)
T_bottom   = 15.0 # fixed bottom temperature (°C)

# Create an array of temperatures, one per layer, all starting at T_initial
T = np.ones(num_layers) * T_initial

# Fix the boundary conditions (surface and bottom never change temperature)
T[0]  = T_surface # set the surface layer
T[-1] = T_bottom #set the bottom layer

We also need to define how time will evolve through the model. Specifically, the time step with which we'll update the temperature and the number of time steps we'll run the model for. Time = 0 will represent the initial model.

In [ ]:
dt = 600.0 # time step (seconds) -- 10 minutes
num_steps = 6*24*20 # number of time steps for 20 days assuming a 10 minute time step

We'll also set up an output variable, where we can record the model temperatures through time for plotting after the model has finished running.

In [ ]:
T_output = [] #create an empty list
T_output.append(T.copy()) #record the initial temperatures with a copy of T

In [ ]:
for step in range(1, num_steps + 1):

    #current time in seconds
    t = step * dt

    # Surface temperature now varies with time as a sine wave
    T[0] = T_mean + A * np.sin(2 * np.pi * t / P)

    T_new = T.copy() # start with current temperatures

    # Update every interior layer (not the surface or bottom -- those are fixed)
    for i in range(1, num_layers - 1):

        # The update rule:
        # If neighbours are warmer on average, this layer warms up a little.
        # If neighbours are cooler on average, this layer cools down a little.
        T_new[i] = T[i] + kappa * (dt / dz**2) * (T[i+1] - 2*T[i] + T[i-1])

    T = T_new #set the temperature to T_new before looping around again
    T_output.append(T.copy()) #record the current temperatures with a copy of T

We can now plot the model results. As you plot the steps you'll see how the temperature evolves through time and how this is different in each layer.

In [ ]:
plot_step = 350 #choose which time step to plot
plot_depth = np.arange(0, depth, dz) #depth of each layer
plt.plot(T_output[plot_step],plot_depth,'ok-') #plot layer temperature versus depth
plt.xlabel('Temperature (°C)') #label the x-axis
plt.ylabel('Soil depth (m)') #label the y-axis
plt.ylim(bottom=-0.05) #limit the y-axis
plt.gca().invert_yaxis() #switch the direction of the y-axis
plt.minorticks_on()

We can also plot how the temperaure of a given layer changes through time.

The soil starts at a uniform 15°C everywhere. When the oscillating surface temperature begins, the first few days of warming pulses penetrate downward and add heat to the soil column. But the cooling pulses at night do not penetrate as deeply.

So the upper-to-middle layers temporarily accumulate more heat than they will hold at true equilibrium. Over successive days, the nightly cooling pulses gradually draw that excess heat back out, and the profile settles into a true repeating cycle where exactly as much heat enters during the warm phase as leaves during the cool phase.

Importantly, the soil needs several cycles to forget its initial starting condition (15°C in every layer) and settle into the true periodic equilibrium. The time required to forget the inital conditions and for the model to reach equilibrium is often referred to as the "spin-up" period (which will vary according to the model setup and parameters).

In [ ]:
depth_index = 10   # index of the layer you want to plot
time_series = [T[depth_index] for T in T_output] #bit of Python magic to extract the values
time_plot = np.arange(0, num_steps * dt + 1, dt)/(60*60*24) #time in days
plt.plot(time_plot,time_series) #plot the temperature over time for the given layer
plt.xlabel('Elapsed time (days)') #label the y-axis
plt.ylabel('Temperature (°C)') #label the x-axis
plt.minorticks_on()